# Execução automática de prompts em LLMs locais (Ollama)

**Objetivo:** ler prompts de um arquivo (`inputs.json`), enviar cada prompt a cada modelo local do Ollama, salvar cada resposta em um arquivo Markdown e manter um CSV consolidado com o registro de todas as execuções (inclusive as que falharam).

**Como usar:** com o Ollama instalado e aberto, execute as células **de cima para baixo** (botão *Run All*). A primeira execução de cada modelo pode demorar vários minutos, pois o modelo precisa ser carregado na memória.

**Onde está o código das funções?** No arquivo `pipeline.py`, ao lado deste notebook. Este notebook mostra a ordem das etapas e, em algumas células, imprime o código das funções para você ler e explicar.

**Fluxo geral**

1. Verificar se o Ollama está funcionando e se os modelos estão instalados.
2. Ler e validar `inputs.json`.
3. Para cada prompt e para cada modelo: enviar o prompt, medir o tempo e receber a resposta.
4. Salvar um `.md` por execução e uma linha no `outputs/resultados.csv`.
5. Consultar um resumo dos resultados.

## 1. Importações

- `inspect` permite exibir o código-fonte de uma função (usado nas células "Ver o código").
- `pipeline` é o nosso módulo (arquivo `pipeline.py`) com as funções do projeto.
- `Path` (da biblioteca `pathlib`) representa caminhos de arquivos e pastas de forma que funciona igual no Windows, Linux e macOS.

In [1]:
import inspect
from pathlib import Path

import pipeline


def mostrar_codigo(*funcoes):
    """Imprime o código-fonte das funções recebidas (útil para estudar e apresentar)."""
    for funcao in funcoes:
        print(inspect.getsource(funcao))

## 2. Configuração

Tudo o que você pode querer mudar fica **nesta célula**:

- `MODELOS`: lista de modelos do experimento. Para trocar um modelo, edite só esta lista. Os nomes precisam ser **exatamente** as tags mostradas por `ollama list`.
- `OPCOES_GERACAO`: parâmetros enviados ao Ollama. `temperature = 0` reduz a aleatoriedade e `seed` fixa a semente do sorteio. Isso **ajuda** a repetir resultados, mas **não garante** respostas idênticas.
- `TIMEOUT_S`: tempo máximo (em segundos) de espera por cada resposta.
- `PARAR_SE_MODELO_AUSENTE`: se `True`, o notebook para quando algum modelo não estiver instalado. Se `False`, ele segue e registra falha para esse modelo no CSV.

In [2]:
# Pasta do projeto = pasta onde está este notebook (o VS Code usa essa pasta por padrão).
PASTA_PROJETO = Path.cwd()
ARQUIVO_INPUTS = PASTA_PROJETO / "inputs.json"   # prompts de entrada
PASTA_SAIDA = PASTA_PROJETO / "outputs"          # resultados (.md e resultados.csv)

URL_OLLAMA = "http://localhost:11434"            # endereço do servidor local do Ollama

MODELOS = [
    "qwen3.5:9b",
    "deepseek-r1:8b",
    "llama3.1:8b",
]

OPCOES_GERACAO = {"temperature": 0, "seed": 42}
TIMEOUT_S = 600
PARAR_SE_MODELO_AUSENTE = True

print("Pasta do projeto:", PASTA_PROJETO)
print("Modelos do experimento:", MODELOS)

Pasta do projeto: c:\Users\oxims\Downloads\files
Modelos do experimento: ['qwen3.5:9b', 'deepseek-r1:8b', 'llama3.1:8b']


## 3. Verificação do Ollama

O Ollama roda como um serviço local que responde a requisições HTTP em `http://localhost:11434`. Esta célula faz uma pergunta simples a ele (*quais modelos você tem?*). Se não houver resposta, a mensagem de erro explica se o Ollama não está instalado ou se só não foi iniciado.

In [3]:
modelos_instalados = pipeline.listar_modelos_instalados(URL_OLLAMA)

print("Ollama respondendo. Modelos instalados:")
for nome in modelos_instalados:
    print(" -", nome)

ErroPipeline: O Ollama está instalado, mas o serviço não respondeu em http://localhost:11434. Abra o aplicativo Ollama ou execute 'ollama serve' em um terminal separado.

## 4. Verificação dos modelos escolhidos

Comparamos `MODELOS` com a lista de instalados, **sem alterar nenhum nome**. Se faltar algum, a célula mostra tags parecidas que estão instaladas (se houver) e o comando para instalar.

> **Diferença importante:** `pip install` instala bibliotecas **Python**. `ollama pull` baixa **modelos** (arquivos de vários gigabytes) para o Ollama. São coisas independentes.

In [4]:
disponiveis, ausentes = pipeline.verificar_modelos(MODELOS, modelos_instalados)

for modelo in disponiveis:
    print(f"OK        {modelo}")
for modelo, parecidos in ausentes.items():
    print(f"AUSENTE   {modelo}")
    if parecidos:
        print(f"          Tags parecidas instaladas: {', '.join(parecidos)}")
    print(f"          Para instalar: ollama pull {modelo}   (confira a tag em https://ollama.com/library)")

if ausentes and PARAR_SE_MODELO_AUSENTE:
    raise RuntimeError(
        "Há modelos ausentes. Instale-os, ajuste a lista MODELOS ou defina PARAR_SE_MODELO_AUSENTE = False."
    )

OK        qwen3.5:9b
OK        deepseek-r1:8b
OK        llama3.1:8b


## 5. Leitura e validação de `inputs.json`

O arquivo JSON contém uma **lista** de prompts. Cada prompt é um **objeto** com `id`, `titulo` e `texto`. Ao ler o arquivo, o Python converte:

| No JSON | No Python |
|---|---|
| lista `[ ... ]` | `list` |
| objeto `{ ... }` | `dict` (dicionário: pares chave → valor) |
| texto `"..."` | `str` |

Para adicionar um prompt novo, basta acrescentar mais um objeto à lista no arquivo. O texto **não é alterado** pelo programa.

In [5]:
prompts = pipeline.carregar_prompts(ARQUIVO_INPUTS)

print(f"{len(prompts)} prompt(s) carregado(s):")
for prompt in prompts:
    print(f" - {prompt['id']}: {prompt['titulo']}  ({len(prompt['texto'])} caracteres)")

print()
print("Tipo do conjunto:", type(prompts).__name__, "| tipo de cada item:", type(prompts[0]).__name__)
print("Chaves de cada prompt:", list(prompts[0].keys()))

3 prompt(s) carregado(s):
 - PROMPT-001: Script de automatização de prompts  (162 caracteres)
 - PROMPT-002: cálulo simples  (38 caracteres)
 - PROMPT-003: Teste de prompt  (80 caracteres)

Tipo do conjunto: list | tipo de cada item: dict
Chaves de cada prompt: ['id', 'titulo', 'texto']


## 6. Função que executa um prompt em um modelo

`executar_prompt_em_modelo` recebe **argumentos** (id do prompt, texto, modelo, opções, timeout) e **retorna** um dicionário com o resultado. Passos:

1. Monta o corpo da requisição (modelo, prompt, `stream: False`, opções).
2. Envia um `POST` para `/api/generate` e inicia o cronômetro (`time.perf_counter`).
3. Interpreta a resposta e extrai texto e metadados (tokens, duração).
4. Em caso de erro (sem conexão, timeout, modelo inexistente, resposta vazia), registra a mensagem em vez de interromper o programa.

A célula abaixo é opcional: ela apenas imprime o código para leitura.

In [6]:
mostrar_codigo(pipeline.executar_prompt_em_modelo)

def executar_prompt_em_modelo(id_prompt, texto_prompt, modelo,
                              url=URL_PADRAO_OLLAMA, opcoes=None, timeout_s=600):
    """Envia UM prompt a UM modelo e devolve um dicionário com tudo o que aconteceu.

    Entradas (argumentos):
      id_prompt    - identificador do prompt (só para registrar no resultado);
      texto_prompt - texto integral enviado ao modelo, sem nenhuma alteração;
      modelo       - tag exata do modelo no Ollama (ex.: "llama3.1:8b");
      opcoes       - parâmetros de geração (ex.: {"temperature": 0, "seed": 42});
      timeout_s    - tempo máximo de espera pela resposta, em segundos.

    Saída (retorno): dicionário com status "sucesso" ou "falha". A função NÃO levanta erro
    para problemas esperados (modelo ausente, timeout...): ela os registra em
    "mensagem_erro". Assim, quem chama (o laço) pode seguir para a próxima execução.
    """
    opcoes = dict(opcoes or {})

    # O dicionário começa como "falha"; só viramos "sucesso" d

## 7. Funções de armazenamento

- `salvar_markdown` cria `outputs/<prompt>/<modelo>__<id_da_execução>.md`. Usa o modo `"x"` do `open()`, que **recusa** sobrescrever um arquivo existente.
- `registrar_no_csv` acrescenta uma linha ao `outputs/resultados.csv` (modo `"a"`, *append*), criando o cabeçalho na primeira vez.
- O **ID da execução** tem a forma `EXEC-AAAAMMDD-HHMMSS-NNN`: data/hora de início do lote + contador. Lotes diferentes nunca repetem IDs.

In [7]:
mostrar_codigo(pipeline.salvar_markdown, pipeline.registrar_no_csv, pipeline.gerar_id_lote, pipeline.gerar_id_execucao)

def salvar_markdown(resultado, pasta_saida):
    """Grava o Markdown da execução em outputs/<prompt>/<modelo>__<id_execucao>.md.

    Devolve o caminho relativo à pasta_saida (é o que vai para o CSV).

    Proteção contra sobrescrita: o modo "x" do open() só cria arquivos NOVOS; se o arquivo
    já existir, o Python levanta FileExistsError em vez de apagar o conteúdo antigo.
    """
    pasta_saida = Path(pasta_saida)
    pasta_prompt = pasta_saida / criar_slug(resultado["id_prompt"])
    pasta_prompt.mkdir(parents=True, exist_ok=True)  # cria a pasta se necessário; não falha se já existir

    nome_arquivo = f"{criar_slug(resultado['modelo'])}__{resultado['id_execucao']}.md"
    caminho = pasta_prompt / nome_arquivo

    # newline="\n" evita que o Windows converta as quebras de linha do texto original.
    with open(caminho, "x", encoding="utf-8", newline="\n") as arquivo:
        arquivo.write(montar_markdown(resultado))
    return caminho.relative_to(pasta_saida).as_posix()

def reg

## 8. Função que coordena o experimento

`processar_experimentos` usa dois `for` aninhados: para **cada prompt**, percorre **cada modelo**, de forma **sequencial** (uma execução por vez). Cada execução é independente: o mesmo texto é enviado separadamente a cada modelo. Uma falha é registrada e o laço continua.

A parte que fala com o Ollama (`executar_tarefa`) está separada da parte que grava em disco; isso facilita uma futura versão paralela.

In [8]:
mostrar_codigo(pipeline.processar_experimentos, pipeline.executar_tarefa)

def processar_experimentos(prompts, modelos, pasta_saida, url=URL_PADRAO_OLLAMA,
                           opcoes=None, timeout_s=600, id_lote=None):
    """Executa TODOS os prompts em TODOS os modelos, um de cada vez, e grava os resultados.

    Ordem: para cada prompt, percorre todos os modelos. Se houver 5 prompts e 3 modelos,
    são 15 execuções independentes, cada uma com o texto original do prompt.
    Falhas são registradas e o laço continua; nada interrompe as demais execuções.
    """
    id_lote = id_lote or gerar_id_lote()
    pasta_saida = Path(pasta_saida)
    caminho_csv = pasta_saida / "resultados.csv"

    # Primeiro montamos a lista de tarefas (cada uma é um dicionário). Dois "for" aninhados
    # geram todas as combinações prompt x modelo, e o contador dá o número de cada execução.
    tarefas = []
    numero = 0
    for prompt in prompts:
        for modelo in modelos:
            numero += 1
            tarefas.append({
                "id_execucao": gerar_id_exec

## 9. Execução do pipeline

Esta é a célula que realmente chama os modelos. Acompanhe o progresso na saída. **Pode demorar bastante**, principalmente na primeira vez e com modelos de raciocínio. Se interromper (botão *Interrupt*), o que já foi concluído permanece salvo.

In [9]:
resultados = pipeline.processar_experimentos(
    prompts=prompts,
    modelos=MODELOS,
    pasta_saida=PASTA_SAIDA,
    url=URL_OLLAMA,
    opcoes=OPCOES_GERACAO,
    timeout_s=TIMEOUT_S,
)

Lote 20261028-195500: 3 prompt(s) x 3 modelo(s) = 9 execução(ões).
[1/9] PROMPT-001 x qwen3.5:9b ... (aguardando resposta)
   -> sucesso em 156.596 s
[2/9] PROMPT-001 x deepseek-r1:8b ... (aguardando resposta)
   -> sucesso em 101.681 s
[3/9] PROMPT-001 x llama3.1:8b ... (aguardando resposta)
   -> sucesso em 84.717 s
[4/9] PROMPT-002 x qwen3.5:9b ... (aguardando resposta)
   -> sucesso em 166.272 s
[5/9] PROMPT-002 x deepseek-r1:8b ... (aguardando resposta)
   -> sucesso em 90.858 s
[6/9] PROMPT-002 x llama3.1:8b ... (aguardando resposta)
   -> sucesso em 97.467 s
[7/9] PROMPT-003 x qwen3.5:9b ... (aguardando resposta)
   -> sucesso em 155.271 s
[8/9] PROMPT-003 x deepseek-r1:8b ... (aguardando resposta)
   -> sucesso em 98.929 s
[9/9] PROMPT-003 x llama3.1:8b ... (aguardando resposta)
   -> sucesso em 83.796 s


## 10. Verificação dos arquivos gerados

Conferimos se cada execução deste lote tem seu arquivo Markdown e se o CSV existe.

In [10]:
sem_arquivo = [r["id_execucao"] for r in resultados
               if not r["arquivo_markdown"] or not (PASTA_SAIDA / r["arquivo_markdown"]).exists()]
sucessos = sum(1 for r in resultados if r["status"] == "sucesso")

print(f"Execuções neste lote: {len(resultados)} | sucessos: {sucessos} | falhas: {len(resultados) - sucessos}")
print(f"Arquivos Markdown encontrados: {len(resultados) - len(sem_arquivo)} de {len(resultados)}")
if sem_arquivo:
    print("Sem arquivo Markdown:", ", ".join(sem_arquivo))
print("CSV consolidado existe:", (PASTA_SAIDA / "resultados.csv").exists())

Execuções neste lote: 9 | sucessos: 9 | falhas: 0
Arquivos Markdown encontrados: 9 de 9
CSV consolidado existe: True


## 11. Consulta e resumo dos resultados

O CSV acumula **todas** as rodadas já feitas (nada é apagado). Abaixo: o que aconteceu neste lote, um resumo por modelo (considerando todo o histórico) e o início de um resultado em Markdown.

In [11]:
print("Resultados deste lote:\n")
pipeline.mostrar_tabela(
    resultados,
    ["id_execucao", "id_prompt", "modelo", "status", "tempo_execucao_s", "tokens_gerados", "arquivo_markdown"],
)

historico = pipeline.ler_resultados_csv(PASTA_SAIDA / "resultados.csv")
print(f"\nResumo por modelo (histórico completo: {len(historico)} execuções no CSV):\n")
pipeline.mostrar_tabela(
    pipeline.resumir_por_modelo(historico),
    ["modelo", "execucoes", "sucessos", "falhas", "tempo_medio_s"],
)

Resultados deste lote:

id_execucao              | id_prompt  | modelo         | status  | tempo_execucao_s | tokens_gerados | arquivo_markdown                                      
-------------------------+------------+----------------+---------+------------------+----------------+-------------------------------------------------------
EXEC-20261028-195500-001 | PROMPT-001 | qwen3.5:9b     | sucesso | 156.596          | 1166           | prompt_001/qwen3_5_9b__EXEC-20261028-195500-001.md    
EXEC-20261028-195500-002 | PROMPT-001 | deepseek-r1:8b | sucesso | 101.681          | 1092           | prompt_001/deepseek_r1_8b__EXEC-20261028-195500-002.md
EXEC-20261028-195500-003 | PROMPT-001 | llama3.1:8b    | sucesso | 84.717           | 239            | prompt_001/llama3_1_8b__EXEC-20261028-195500-003.md   
EXEC-20261028-195500-004 | PROMPT-002 | qwen3.5:9b     | sucesso | 166.272          | 522            | prompt_002/qwen3_5_9b__EXEC-20261028-195500-004.md    
EXEC-20261028-195500-005 | P

In [12]:
# Mostra as primeiras linhas do Markdown da primeira execução bem-sucedida deste lote.
primeira = next((r for r in resultados if r["status"] == "sucesso"), None)
if primeira:
    texto_md = (PASTA_SAIDA / primeira["arquivo_markdown"]).read_text(encoding="utf-8")
    print("\n".join(texto_md.splitlines()[:40]))
else:
    print("Nenhuma execução bem-sucedida neste lote. Veja a coluna mensagem_erro no CSV.")

# Resultado da execução

## Identificação

- ID da execução: EXEC-20261028-195500-001
- ID do prompt: PROMPT-001
- Título do prompt: Script de automatização de prompts
- Modelo (tag exata): `qwen3.5:9b`
- Data e hora: 2026-10-28T19:55:00-03:00
- Status: sucesso
- Tempo de execução: 156.596 s

## Parâmetros de geração

- temperature: 0
- seed: 42
- timeout (s): 600

## Metadados informados pelo Ollama

- Tokens do prompt: 45
- Tokens gerados: 1166
- Duração total no Ollama (s): 154.551
- Motivo de término: stop

## Input

```
Se eu utilizar um script de automatizar o envio e recebimento de outputs utilizando o ollama, eu consigo fazer isso sem estar com ele aberto? explique brevemente.
```

## Raciocínio do modelo (thinking)

Thinking Process:

1.  **Analyze the Request:**
    *   **Topic:** Automating input/output (sending and receiving) using Ollama.
    *   **Question:** Can this be done without keeping the Ollama application/interface open?
    *   **Language:** Portuguese.
    *   